In [66]:
!pip install tiktoken > \dev\null

In [97]:
import re
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken
import urllib.request
from importlib.metadata import version

tokenizer = tiktoken.get_encoding('gpt2')

In [87]:
#  DOWNLOAD THE DATASET AND SAVE IT TO "the-verdict.txt"
def download_data(url: str = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"):
  file_path = "the-verdict.txt"
  urllib.request.urlretrieve(url, file_path)
  with open('the-verdict.txt', 'r', encoding='utf-8') as f:
    return f.read()

raw_text = download_data()

#  PREPROCESSING
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

#  DEFINE VOCABULARY
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)  # 1130
vocab = {token:integer for integer, token in  enumerate(all_words)}
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer, token in enumerate(all_tokens)}

#  ENCODING TEXT WITH BPE tokenizer
enc_text = tokenizer.encode(raw_text)
len(enc_text)  # 5145

5145

In [88]:
class SimpleTokenizer:
  def __init__(self, vocab: dict) -> None:
    self.str_to_int = vocab
    self.int_to_str = {i:s for s, i in vocab.items()}

  def encode(self, text: str):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]

    preprocessed = [item if item in self.str_to_int
                    else "<|unk|>" for item in preprocessed]

    ids = [self.str_to_int[s] for s in preprocessed]

    return ids

  def decode(self, ids: list[int]):
    text = " ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

#  tokenizer = SimpleTokenizer(vocab)
#  print(tokenizer.decode(tokenizer.encode("I like computer.<|endoftext|>")))

In [117]:
class GPTDatasetV1(Dataset):
  def __init__(self, txt: str, tokenizer: object, max_lenght: int, stride: int):
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

    for i in range(0, len(token_ids) - max_lenght, stride):
      input_chunk = token_ids[i:i+max_lenght]
      target_chunk = token_ids[i+1:i+max_lenght+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [118]:
def create_dataloader_v1(txt, batch_size=4, max_lenght=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
  tokenizer = tiktoken.get_encoding('gpt2')
  dataset = GPTDatasetV1(txt, tokenizer, max_lenght, stride)
  dataloader = DataLoader(
      dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers
  )
  return dataloader

In [120]:
max_lenght = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_lenght=max_lenght, stride=4, shuffle=False)

In [130]:
vocab_size = 50257
output_dim = 256

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
token_embeddings = token_embedding_layer(inputs)

context_lenght = max_lenght
pos_embedding_layer = torch.nn.Embedding(context_lenght, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_lenght))

input_embeddings = token_embeddings + pos_embeddings

print(input_embeddings.shape)

torch.Size([8, 4, 256])
